In [1]:
import pandas as pd
import numpy as np
df = pd.read_csv('data.csv')

In [2]:
df['Review text'] = df['Review text'].apply(lambda x: str(x).replace("READ MORE", "").strip())
def classify_sentiment(rating):
    if rating <= 3:
        return 'Negative'
    else:
        return 'Positive'
df['Sentiment'] = df['Ratings'].apply(classify_sentiment)
label_mapping = {'Negative': 0, 'Positive': 1}
df['Sentiment_Label'] = df['Sentiment'].map(label_mapping)

In [3]:
print(f"Total reviews: {df.shape[0]}")
print("\nClass Distribution:")
print(df['Sentiment'].value_counts())

df[['Review text', 'Ratings', 'Sentiment']].head()

Total reviews: 8518

Class Distribution:
Sentiment
Positive    6826
Negative    1692
Name: count, dtype: int64


,Review text,Ratings,Sentiment
0,"Nice product, good quality, but price is now r...",4,Positive
1,They didn't supplied Yonex Mavis 350. Outside ...,1,Negative
2,Worst product. Damaged shuttlecocks packed in ...,1,Negative
3,"Quite O. K. , but nowadays the quality of the...",3,Negative
4,Over pricedJust â?¹620 ..from retailer.I didn'...,1,Negative


In [4]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/souryadipmallick/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/souryadipmallick/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /Users/souryadipmallick/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [5]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    words = text.split()
    
    clean_words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    return " ".join(clean_words)

df['Cleaned_Review'] = df['Review text'].apply(preprocess_text)
df[['Review text', 'Cleaned_Review', 'Sentiment']].head()

,Review text,Cleaned_Review,Sentiment
0,"Nice product, good quality, but price is now r...",nice product good quality price rising bad sig...,Positive
1,They didn't supplied Yonex Mavis 350. Outside ...,didnt supplied yonex mavis outside cover yonex...,Negative
2,Worst product. Damaged shuttlecocks packed in ...,worst product damaged shuttlecock packed new b...,Negative
3,"Quite O. K. , but nowadays the quality of the...",quite k nowadays quality cork like year back u...,Negative
4,Over pricedJust â?¹620 ..from retailer.I didn'...,pricedjust retaileri didnt understand wat adva...,Negative


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score

# X = cleaned text, y = sentiment label
X = df['Cleaned_Review']
y = df['Sentiment_Label']

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [8]:
tfidf = TfidfVectorizer(max_features=32000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

model = LogisticRegression(class_weight='balanced', max_iter=1000)
model.fit(X_train_tfidf, y_train)

y_pred = model.predict(X_test_tfidf)
f1 = f1_score(y_test, y_pred, average='macro')

In [9]:
print(f"Model F1-Score (Macro): {f1:.4f}")
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))

Model F1-Score (Macro): 0.7716

Classification Report:

              precision    recall  f1-score   support

    Negative       0.58      0.72      0.64       325
    Positive       0.93      0.88      0.90      1379

    accuracy                           0.85      1704
   macro avg       0.75      0.80      0.77      1704
weighted avg       0.86      0.85      0.85      1704



In [10]:
import pickle

with open('sentiment_model.pkl', 'wb') as f:
    pickle.dump(model, f)

with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)